# NB20: Dataset Collection, Alignment, Annotation & Balancing

Implements the full data pipeline from plan.md:
1. Collect 5 datasets
2. Align labels to unified schema
3. Deduplicate
4. Entity & sentiment annotation (agent-labeled)
5. Balance sources
6. Train/val/test split
7. Quality validation
8. Push to HuggingFace

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import numpy as np
import re
import os
import glob

OUTPUT_DIR = "../data/processed"
ANNOTATION_DIR = f"{OUTPUT_DIR}/entity_annotations"
os.makedirs(ANNOTATION_DIR, exist_ok=True)

## Phase 1: Dataset Collection & Schema Alignment

In [ ]:
LABEL_MAPS = {
    "nosible": {
        "positive": "POSITIVE",
        "negative": "NEGATIVE",
        "neutral": "NEUTRAL",
    },
    "timkoornstra": {
        0: "NEUTRAL",
        1: "POSITIVE",
        2: "NEGATIVE",
    },
    "finsent": {
        "neutral": "NEUTRAL",
        "positive": "POSITIVE",
        "negative": "NEGATIVE",
    },
    "aiera": {
        "positive": "POSITIVE",
        "negative": "NEGATIVE",
        "neutral": "NEUTRAL",
    },
    "subjectiveqa": {
        0: "NEGATIVE",
        1: "NEUTRAL",
        2: "POSITIVE",
    },
}

In [ ]:
# Dataset 1: NOSIBLE (100K)
ds = load_dataset("NOSIBLE/financial-sentiment", split="train")
df_nosible = ds.to_pandas()

df_nosible = df_nosible.rename(columns={"label": "raw_label"})
df_nosible["label"] = df_nosible["raw_label"].map(LABEL_MAPS["nosible"])
df_nosible["source"] = "nosible"
df_nosible["source_domain"] = "financial_news"
df_nosible["label_confidence"] = "llm_consensus"

df_nosible = df_nosible[df_nosible["text"].str.len() >= 20]
df_nosible = df_nosible[df_nosible["text"].str.len() <= 2000]
df_nosible = df_nosible.dropna(subset=["text", "label"])

df_nosible = df_nosible[["text", "label", "source", "source_domain", "label_confidence"]]
print(f"NOSIBLE: {len(df_nosible)} rows")
print(df_nosible["label"].value_counts())

In [ ]:
# Dataset 2: TimKoornstra Tweets (38K)
ds = load_dataset("TimKoornstra/financial-tweets-sentiment", split="train")
df_tweets = ds.to_pandas()

df_tweets = df_tweets.rename(columns={"tweet": "text", "sentiment": "raw_label"})
df_tweets["label"] = df_tweets["raw_label"].map(LABEL_MAPS["timkoornstra"])
df_tweets["source"] = "timkoornstra_tweets"
df_tweets["source_domain"] = "social_media"
df_tweets["label_confidence"] = "human_aggregated"

df_tweets = df_tweets[df_tweets["text"].str.len() >= 10]
df_tweets = df_tweets.dropna(subset=["text", "label"])

df_tweets = df_tweets[["text", "label", "source", "source_domain", "label_confidence"]]
print(f"TimKoornstra: {len(df_tweets)} rows")
print(df_tweets["label"].value_counts())

In [ ]:
# Dataset 3: FinanceMTEB FinSent (10K)
ds = load_dataset("FinanceMTEB/FinSent")
df_finsent = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()])

df_finsent["label"] = df_finsent["label_text"].map(LABEL_MAPS["finsent"])
df_finsent["source"] = "financemteb_finsent"
df_finsent["source_domain"] = "analyst_reports"
df_finsent["label_confidence"] = "human"

df_finsent = df_finsent[df_finsent["text"].str.len() >= 10]
df_finsent = df_finsent.dropna(subset=["text", "label"])

df_finsent = df_finsent[["text", "label", "source", "source_domain", "label_confidence"]]
print(f"FinSent: {len(df_finsent)} rows")
print(df_finsent["label"].value_counts())

In [ ]:
# Dataset 4: Aiera Transcript Sentiment (700)
ds = load_dataset("Aiera/aiera-transcript-sentiment", split="test")
df_aiera = ds.to_pandas()

df_aiera = df_aiera.rename(columns={"transcript": "text", "sentiment": "raw_label"})
df_aiera["label"] = df_aiera["raw_label"].map(LABEL_MAPS["aiera"])
df_aiera["source"] = "aiera_transcripts"
df_aiera["source_domain"] = "earnings_calls"
df_aiera["label_confidence"] = "human"

df_aiera = df_aiera[["text", "label", "source", "source_domain", "label_confidence"]]
print(f"Aiera: {len(df_aiera)} rows")
print(df_aiera["label"].value_counts())

In [ ]:
# Dataset 5: SubjECTive-QA (2.7K earnings call QA)
ds = load_dataset("gtfintechlab/SubjECTive-QA", "5768")
df_sqqa = pd.concat([ds["train"].to_pandas(), ds["val"].to_pandas(), ds["test"].to_pandas()])

df_sqqa = df_sqqa.rename(columns={"ANSWER": "text", "OPTIMISTIC": "raw_label"})
df_sqqa["label"] = df_sqqa["raw_label"].map(LABEL_MAPS["subjectiveqa"])
df_sqqa["source"] = "subjectiveqa"
df_sqqa["source_domain"] = "earnings_calls"
df_sqqa["label_confidence"] = "human"

df_sqqa = df_sqqa[df_sqqa["text"].str.len() >= 20]
df_sqqa = df_sqqa.dropna(subset=["text", "label"])

df_sqqa = df_sqqa[["text", "label", "source", "source_domain", "label_confidence"]]
print(f"SubjECTive-QA: {len(df_sqqa)} rows")
print(df_sqqa["label"].value_counts())

In [ ]:
# Combine all datasets
df_all = pd.concat([
    df_nosible,
    df_tweets,
    df_finsent,
    df_aiera,
    df_sqqa,
], ignore_index=True)

print(f"Total combined: {len(df_all)} rows")
print(f"\nPer source:\n{df_all['source'].value_counts()}")
print(f"\nPer label:\n{df_all['label'].value_counts()}")

df_all.to_parquet(f"{OUTPUT_DIR}/all_raw_combined.parquet", index=False)
print(f"\nSaved to {OUTPUT_DIR}/all_raw_combined.parquet")

## Phase 2: Deduplication

In [ ]:
# Level 1: Exact text dedup
df_all["text_norm"] = df_all["text"].apply(lambda t: re.sub(r"\s+", " ", str(t).strip().lower()))
n_before = len(df_all)
df_all = df_all.drop_duplicates(subset=["text_norm"], keep="first")
print(f"Exact dedup: {n_before} -> {len(df_all)} ({n_before - len(df_all)} removed)")
df_all = df_all.drop(columns=["text_norm"]).reset_index(drop=True)

In [ ]:
# Level 2: Semantic near-duplicate detection
from sentence_transformers import SentenceTransformer
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    df_all["text"].tolist(),
    batch_size=512,
    show_progress_bar=True,
    normalize_embeddings=True,
)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

THRESHOLD = 0.95
K = 10
distances, indices = index.search(embeddings.astype(np.float32), K)

to_remove = set()
for i in range(len(df_all)):
    if i in to_remove:
        continue
    for j_idx, dist in zip(indices[i], distances[i]):
        if j_idx > i and dist > THRESHOLD and j_idx not in to_remove:
            to_remove.add(j_idx)

n_before = len(df_all)
df_all = df_all.drop(index=df_all.index[list(to_remove)]).reset_index(drop=True)
print(f"Semantic dedup: {n_before} -> {len(df_all)} ({len(to_remove)} removed)")
print(f"\nPost-dedup per source:\n{df_all['source'].value_counts()}")

df_all.to_parquet(f"{OUTPUT_DIR}/all_deduped.parquet", index=False)
print(f"\nSaved to {OUTPUT_DIR}/all_deduped.parquet")

## Phase 3: Entity & Aspect Sentiment Annotation (Agent-Labeled)

Prepare batches for agent annotation. The agent reads each batch, determines entity + entity_sentiment, and writes output CSVs.

In [ ]:
# Prepare annotation batches
BATCH_SIZE = 500
n_batches = (len(df_all) + BATCH_SIZE - 1) // BATCH_SIZE
for i in range(n_batches):
    start = i * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(df_all))
    batch = df_all.iloc[start:end][["text", "label"]].copy()
    batch.to_csv(f"{ANNOTATION_DIR}/batch_{i:04d}_input.csv", index=True)
print(f"Prepared {n_batches} batches of ~{BATCH_SIZE} rows in {ANNOTATION_DIR}/")

In [ ]:
# Assemble annotations after agent completes all batches
output_files = sorted(glob.glob(f"{ANNOTATION_DIR}/batch_*_output.csv"))
print(f"Found {len(output_files)} annotation output files")

annotations = pd.concat([pd.read_csv(f, index_col=0) for f in output_files])
df_all["entity"] = annotations["entity"].values
df_all["entity_sentiment"] = annotations["entity_sentiment"].values

assert df_all["entity"].notna().all(), "Missing entity annotations"
assert df_all["entity_sentiment"].isin(["POSITIVE", "NEGATIVE", "NEUTRAL"]).all()
print(f"Entity coverage (non-NONE): {(df_all['entity'] != 'NONE').mean():.1%}")
print(f"\nTop 20 entities:\n{df_all['entity'].value_counts().head(20)}")
print(f"\nEntity sentiment distribution:\n{df_all['entity_sentiment'].value_counts()}")
print(f"\nAgreement with sentence label: {(df_all['entity_sentiment'] == df_all['label']).mean():.1%}")

df_all.to_parquet(f"{OUTPUT_DIR}/all_annotated.parquet", index=False)
print(f"\nSaved to {OUTPUT_DIR}/all_annotated.parquet")

## Phase 4: Balancing

In [ ]:
MAX_PER_SOURCE = 15_000

def balance_dataset(df, max_per_source=MAX_PER_SOURCE, random_state=42):
    source_sizes = df["source"].value_counts()
    print(f"Raw source sizes:\n{source_sizes}\n")

    sampled_dfs = []
    for source in df["source"].unique():
        source_df = df[df["source"] == source]
        if len(source_df) > max_per_source:
            n_per_label = max_per_source // 3
            label_dfs = []
            for label in ["NEGATIVE", "NEUTRAL", "POSITIVE"]:
                label_df = source_df[source_df["label"] == label]
                if len(label_df) >= n_per_label:
                    label_dfs.append(label_df.sample(n=n_per_label, random_state=random_state))
                elif len(label_df) > 0:
                    label_dfs.append(label_df)
            sampled_dfs.append(pd.concat(label_dfs, ignore_index=True))
        else:
            sampled_dfs.append(source_df)

    df_balanced = pd.concat(sampled_dfs, ignore_index=True)
    df_balanced = df_balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df_balanced

df_balanced = balance_dataset(df_all)

print(f"\nFinal balanced dataset: {len(df_balanced)} rows")
print(f"\nSource distribution:\n{df_balanced['source'].value_counts()}")
print(f"\nLabel distribution:\n{df_balanced['label'].value_counts()}")
print(f"\nLabel by source:")
print(df_balanced.groupby(["source", "label"]).size().unstack(fill_value=0))

df_balanced.to_parquet(f"{OUTPUT_DIR}/all_balanced.parquet", index=False)

## Phase 5: Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

strat_col = df_balanced[["source", "label"]].apply(tuple, axis=1)
df_train, df_temp = train_test_split(df_balanced, test_size=0.2, random_state=42, stratify=strat_col)

strat_col_temp = df_temp[["source", "label"]].apply(tuple, axis=1)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=strat_col_temp)

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

df_train.to_parquet(f"{OUTPUT_DIR}/train.parquet", index=False)
df_val.to_parquet(f"{OUTPUT_DIR}/val.parquet", index=False)
df_test.to_parquet(f"{OUTPUT_DIR}/test.parquet", index=False)
print("Saved train/val/test parquets")

## Phase 6: Quality Validation

In [ ]:
def validate_dataset(df, name="dataset"):
    print(f"\n{'='*60}")
    print(f"Validation: {name} ({len(df)} rows)")
    print(f"{'='*60}")

    nulls = df.isnull().sum()
    assert nulls.sum() == 0, f"Found nulls:\n{nulls[nulls > 0]}"
    print("[PASS] No null values")

    valid_labels = {"NEGATIVE", "NEUTRAL", "POSITIVE"}
    assert set(df["label"].unique()) <= valid_labels
    assert set(df["entity_sentiment"].unique()) <= valid_labels
    print("[PASS] All labels valid")

    lengths = df["text"].str.len()
    print(f"  Text length: min={lengths.min()}, median={lengths.median():.0f}, max={lengths.max()}")
    assert lengths.min() >= 10, "Texts too short"
    print("[PASS] Text length range OK")

    source_pcts = df["source"].value_counts(normalize=True)
    assert source_pcts.max() <= 0.40, f"Source {source_pcts.idxmax()} is {source_pcts.max():.1%}"
    print(f"[PASS] Source balance OK (max: {source_pcts.max():.1%})")
    for src, pct in source_pcts.items():
        print(f"  {src}: {pct:.1%}")

    label_pcts = df["label"].value_counts(normalize=True)
    for label, pct in label_pcts.items():
        print(f"  {label}: {pct:.1%}")
    max_deviation = abs(label_pcts - 1/3).max()
    print(f"  Max deviation from 33%: {max_deviation:.1%}")
    print(f"[INFO] Label balance (natural distribution preserved for small sources)")

    n_dupes = df.duplicated(subset=["text"]).sum()
    print(f"  Exact duplicates: {n_dupes}")

    entity_coverage = (df["entity"] != "NONE").mean()
    print(f"  Entity coverage: {entity_coverage:.1%}")

    agreement = (df["entity_sentiment"] == df["label"]).mean()
    print(f"  Entity-sentence sentiment agreement: {agreement:.1%}")

    print(f"\n[ALL PASSED] {name}")

validate_dataset(df_train, "Training Set")
validate_dataset(df_val, "Validation Set")
validate_dataset(df_test, "Test Set")

In [ ]:
# Cross-split leakage check
train_texts = set(df_train["text"])
val_texts = set(df_val["text"])
test_texts = set(df_test["text"])

tv_leak = train_texts & val_texts
tt_leak = train_texts & test_texts
vt_leak = val_texts & test_texts

print(f"Train-Val leakage: {len(tv_leak)}")
print(f"Train-Test leakage: {len(tt_leak)}")
print(f"Val-Test leakage: {len(vt_leak)}")
assert len(tv_leak) == 0 and len(tt_leak) == 0 and len(vt_leak) == 0, "LEAKAGE DETECTED"
print("[PASS] No cross-split leakage")

In [ ]:
# Final dataset card summary
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
for split_name, split_df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"\n{split_name}: {len(split_df)} rows")
print(f"\nSource composition:")
print(df_balanced["source"].value_counts())
print(f"\nLabel distribution:")
print(df_balanced["label"].value_counts())
print(f"\nDomain coverage:")
print(df_balanced["source_domain"].value_counts())
print(f"\nUnique entities: {df_balanced['entity'].nunique()}")
print(f"Entity coverage: {(df_balanced['entity'] != 'NONE').mean():.1%}")

## Phase 7: Push to HuggingFace

In [ ]:
ds_dict = DatasetDict({
    "train": Dataset.from_pandas(df_train, preserve_index=False),
    "validation": Dataset.from_pandas(df_val, preserve_index=False),
    "test": Dataset.from_pandas(df_test, preserve_index=False),
})

print(ds_dict)

ds_dict.push_to_hub(
    "neoyipeng/modernfinbert-training-v2",
    private=False,
)
print("Pushed to HuggingFace!")

In [ ]:
# Verify
ds_verify = load_dataset("neoyipeng/modernfinbert-training-v2")
print(ds_verify)